In [19]:
import requests
from pathlib import Path
import pandas as pd

In [ ]:
url = "https://previous-runs-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 39.2238,
    "longitude": 9.1217,
    "start_date": "2026-07-01",
    "end_date": "2026-07-03",
    "hourly": [
        "temperature_2m_previous_day1",
        "relative_humidity_2m_previous_day1",
        "cloud_cover_previous_day1",
    ],
    "timezone": "UTC",
}

response = requests.get(url, params=params)
response.raise_for_status()

weather = pd.DataFrame(response.json()["hourly"])
weather.head()

,time,temperature_2m_previous_day1,relative_humidity_2m_previous_day1,cloud_cover_previous_day1
0,2026-07-01T00:00,26.1,85,74
1,2026-07-01T01:00,25.6,87,5
2,2026-07-01T02:00,25.2,91,7
3,2026-07-01T03:00,24.7,94,26
4,2026-07-01T04:00,24.5,96,48


In [7]:
locations = {
    "cagliari": (39.2238, 9.1217),
    "oristano": (39.9062, 8.5884),
    "nuoro": (40.3202, 9.3282),
    "sassari": (40.7259, 8.5557),
    "olbia": (40.9236, 9.4964),
}

weather_data = {}

for name, (lat, lon) in locations.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2024-01-01",
        "end_date": "2026-10-05",
        "hourly": [
            "temperature_2m_previous_day1",
            "relative_humidity_2m_previous_day1",
            "cloud_cover_previous_day1",
        ],
    "timezone": "UTC",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    weather_data[name] = pd.DataFrame(response.json()["hourly"])

    print(name, len(weather_data[name]))

cagliari 24216
oristano 24216
nuoro 24216
sassari 24216
olbia 24216


In [8]:
for name, w in weather_data.items():
    tmp = w.copy()
    tmp["time"] = pd.to_datetime(tmp["time"])

    print(f"\n{name.upper()}")
    print("Range:", tmp["time"].min(), "->", tmp["time"].max())
    print("Duplicated timestamps:", tmp["time"].duplicated().sum())
    print("Missing values:")
    print(tmp.isna().sum())


CAGLIARI
Range: 2024-01-01 00:00:00 -> 2026-10-05 23:00:00
Duplicated timestamps: 0
Missing values:
time                                    0
temperature_2m_previous_day1          444
relative_humidity_2m_previous_day1    444
cloud_cover_previous_day1             444
dtype: int64

ORISTANO
Range: 2024-01-01 00:00:00 -> 2026-10-05 23:00:00
Duplicated timestamps: 0
Missing values:
time                                    0
temperature_2m_previous_day1          444
relative_humidity_2m_previous_day1    444
cloud_cover_previous_day1             444
dtype: int64

NUORO
Range: 2024-01-01 00:00:00 -> 2026-10-05 23:00:00
Duplicated timestamps: 0
Missing values:
time                                    0
temperature_2m_previous_day1          444
relative_humidity_2m_previous_day1    444
cloud_cover_previous_day1             444
dtype: int64

SASSARI
Range: 2024-01-01 00:00:00 -> 2026-10-05 23:00:00
Duplicated timestamps: 0
Missing values:
time                                    0
temperature_2m_

In [9]:
w = weather_data["cagliari"].copy()
w["time"] = pd.to_datetime(w["time"])

missing = w[w["temperature_2m_previous_day1"].isna()]

print("First missing:", missing["time"].min())
print("Last missing:", missing["time"].max())

print(
    missing.groupby(missing["time"].dt.to_period("M")).size()
)

First missing: 2024-01-01 00:00:00
Last missing: 2024-01-19 11:00:00
time
2024-01    444
Freq: M, dtype: int64


In [10]:
w = weather_data["cagliari"].copy()
w["time"] = pd.to_datetime(w["time"])

spring_dst = w[
    (w["time"] >= "2024-03-31 00:00") &
    (w["time"] <= "2024-03-31 05:00")
]

autumn_dst = w[
    (w["time"] >= "2024-10-27 00:00") &
    (w["time"] <= "2024-10-27 05:00")
]

print("SPRING DST")
print(spring_dst[["time", "temperature_2m_previous_day1"]].to_string(index=False))

print("\nAUTUMN DST")
print(autumn_dst[["time", "temperature_2m_previous_day1"]].to_string(index=False))

SPRING DST
               time  temperature_2m_previous_day1
2024-03-31 00:00:00                          13.5
2024-03-31 01:00:00                          14.1
2024-03-31 02:00:00                          14.1
2024-03-31 03:00:00                          13.9
2024-03-31 04:00:00                          14.5
2024-03-31 05:00:00                          14.8

AUTUMN DST
               time  temperature_2m_previous_day1
2024-10-27 00:00:00                          20.1
2024-10-27 01:00:00                          19.9
2024-10-27 02:00:00                          19.9
2024-10-27 03:00:00                          19.8
2024-10-27 04:00:00                          19.7
2024-10-27 05:00:00                          19.4


In [11]:
w = weather_data["cagliari"].copy()

w["time"] = pd.to_datetime(w["time"], utc=True)

print("Range:", w["time"].min(), "->", w["time"].max())
print("Duplicates:", w["time"].duplicated().sum())

dt = w["time"].diff()
print(dt.value_counts())

Range: 2024-01-01 00:00:00+00:00 -> 2026-10-05 23:00:00+00:00
Duplicates: 0
time
0 days 01:00:00    24215
Name: count, dtype: int64


In [12]:
weather_utc = {}

for name, w in weather_data.items():
    tmp = w.copy()

    tmp["time"] = pd.to_datetime(tmp["time"], utc=True)
    tmp = tmp.set_index("time")

    tmp = tmp.rename(columns={
        "temperature_2m_previous_day1": f"temperature_{name}",
        "relative_humidity_2m_previous_day1": f"humidity_{name}",
        "cloud_cover_previous_day1": f"cloud_cover_{name}",
    })

    weather_utc[name] = tmp

In [15]:
weather_all = pd.concat(weather_utc.values(), axis=1)

print("Shape:", weather_all.shape)
print("Range:", weather_all.index.min(), "->", weather_all.index.max())
print("Duplicated timestamps:", weather_all.index.duplicated().sum())

weather_all.tail()

Shape: (24216, 15)
Range: 2024-01-01 00:00:00+00:00 -> 2026-10-05 23:00:00+00:00
Duplicated timestamps: 0


,temperature_cagliari,humidity_cagliari,cloud_cover_cagliari,temperature_oristano,humidity_oristano,cloud_cover_oristano,temperature_nuoro,humidity_nuoro,cloud_cover_nuoro,temperature_sassari,humidity_sassari,cloud_cover_sassari,temperature_olbia,humidity_olbia,cloud_cover_olbia
time,,,,,,,,,,,,,,,
2026-10-05 19:00:00+00:00,24.1,71.0,100.0,24.7,69.0,100.0,18.4,92.0,100.0,22.4,69.0,100.0,23.7,77.0,100.0
2026-10-05 20:00:00+00:00,23.7,75.0,86.0,24.1,71.0,83.0,18.2,94.0,93.0,21.7,75.0,100.0,23.4,79.0,98.0
2026-10-05 21:00:00+00:00,23.3,78.0,100.0,23.6,73.0,92.0,18.1,92.0,94.0,20.6,85.0,100.0,23.1,82.0,98.0
2026-10-05 22:00:00+00:00,22.7,81.0,94.0,23.1,75.0,100.0,17.8,91.0,97.0,19.9,90.0,100.0,22.8,84.0,100.0
2026-10-05 23:00:00+00:00,22.2,83.0,100.0,22.6,78.0,100.0,17.5,92.0,100.0,19.6,91.0,100.0,22.4,86.0,100.0


In [16]:
# ISTAT data from 2023

population = {
    "cagliari": 534_219 + 117_780 + 89_155,
    "oristano": 146_785,
    "nuoro": 142_241 + 53_722,
    "sassari": 311_128,
    "olbia": 159_460,
}

total_population = sum(population.values())

population_weights = {
    location: pop / total_population
    for location, pop in population.items()
}

population_weights

{'cagliari': 0.4767827390333807,
 'oristano': 0.09442646784475937,
 'nuoro': 0.12606256714420808,
 'sassari': 0.20014795849442582,
 'olbia': 0.102580267483226}

In [17]:
weather_regional = pd.DataFrame(index=weather_all.index)

# Population-weighted regional weather
weather_regional["temperature_weighted"] = sum(
    weather_all[f"temperature_{location}"] * weight
    for location, weight in population_weights.items()
)

weather_regional["humidity_weighted"] = sum(
    weather_all[f"humidity_{location}"] * weight
    for location, weight in population_weights.items()
)

weather_regional["cloud_cover_weighted"] = sum(
    weather_all[f"cloud_cover_{location}"] * weight
    for location, weight in population_weights.items()
)

# Geographic temperature extremes
temperature_cols = [
    f"temperature_{location}"
    for location in population_weights
]

weather_regional["temperature_min"] = weather_all[temperature_cols].min(axis=1)
weather_regional["temperature_max"] = weather_all[temperature_cols].max(axis=1)

weather_regional.describe()

,temperature_weighted,humidity_weighted,cloud_cover_weighted,temperature_min,temperature_max
count,23772.000000,23772.000000,23772.000000,23772.000000,23772.000000
mean,18.276839,71.326169,47.809407,15.445688,19.981689
std,6.878855,15.518862,34.227524,7.290079,7.038377
min,2.455443,23.514930,0.000000,-0.200000,3.200000
25%,12.659239,59.953626,15.768490,9.500000,14.200000
50%,17.538473,73.954833,44.390520,14.500000,19.100000
75%,23.375841,84.279990,80.541516,20.900000,25.100000
max,38.013569,96.931457,100.000000,35.000000,42.200000


In [20]:
OUTPUT_PATH = Path("../data/processed/sardinia_weather_hourly.parquet")

weather_regional.to_parquet(OUTPUT_PATH)

print(f"Saved to: {OUTPUT_PATH}")
print("Shape:", weather_regional.shape)

Saved to: ../data/processed/sardinia_weather_hourly.parquet
Shape: (24216, 5)
